In [ ]:
import flexlibs
from SIL.LCModel import *
from SIL.LCModel.Core.KernelInterfaces import ITsString, ITsTextProps, ITsStrFactory, ITsPropsFactory, ITsStrBldr, ITsIncStrBldr, ITsPropsBldr, ITsMultiString, FwTextPropType, FwTextPropVar
from SIL.LCModel.Core.Text import TsStringUtils,TsIncStrBldr, TsStrBldr, TsStringComparer
from SIL.LCModel.Core.Cellar import CellarPropertyType, CellarPropertyTypeFilter, GenDate
from SIL.LCModel.Core.WritingSystems import CoreWritingSystemDefinition
from SIL.LCModel.Infrastructure import IFwMetaDataCacheManaged
from System import Guid, String, DateTime, Byte
from docx.shared import Inches, Pt
import unicodedata
import docx
# import tkinter as tk
# from tkinter import ttk

flexlibs.FLExInitialize()

In [6]:
project = flexlibs.FLExProject()
project.OpenProject('tkr-lexicon-flex')
# project.OpenProject('IndonesianTest')
project.ProjectName()
cache = project.project
lp = cache.LangProject
lexicon = lp.LexDbOA
sl = cache.ServiceLocator
mdcm = IFwMetaDataCacheManaged(cache.MetaDataCacheAccessor)
ddbf = cache.DomainDataByFlid
wsv = cache.DefaultVernWs
wsa = cache.DefaultAnalWs

entry = next(iter(lexicon.Entries))


In [7]:
analysis_ws_list = project.GetAllAnalysisWSs()

test_tag = None

# Iterate through each analysis writing system
for ws_tag in analysis_ws_list:
    ws_handle = project.WSHandle(ws_tag)
    ws_name = project.WSUIName(ws_tag)
    
    print(ws_tag, ws_handle, ws_name)
    test_handle = ws_handle
    if ws_tag == 'en':
        wsa = ws_handle



ru 999000007 russian
azj-Latn-AZ 999000002 North Azerbaijani (Latin, Azerbaijan)
en 999000004 English


In [8]:
# returns the form, and if it should split on a punctuation
def get_form(analysis, wsv) -> tuple[str, bool] | None:
    
    class_name = analysis.ClassName
    hvo = analysis.Hvo
    
    try:
        if class_name == "WfiWordform":
            obj = cache.ServiceLocator.GetService(IWfiWordformRepository).GetObject(hvo)
            return (obj.Form.get_String(wsv).Text, False)
            
        elif class_name == "WfiGloss":
            obj = cache.ServiceLocator.GetService(IWfiGlossRepository).GetObject(hvo)
            analysis = obj.Owner
            word_form = analysis.Owner
            wf = cache.ServiceLocator.GetService(IWfiWordformRepository).GetObject(word_form.Hvo)
            return (wf.Form.get_String(wsv).Text, False)
            
        elif class_name == "PunctuationForm":
            obj = cache.ServiceLocator.GetService(IPunctuationFormRepository).GetObject(hvo)
            return (str(obj.Form), True)
            
        elif class_name == "WfiAnalysis":
            obj = cache.ServiceLocator.GetService(IWfiAnalysisRepository).GetObject(hvo)
            if obj.Owner:
                return (obj.Owner.Form.get_String(wsv).Text, False)
            
    except Exception as e:
        print(f"Error with {class_name}: {e}")

In [9]:
def GetAllCustomFields():
    for flid in mdcm.GetFieldIds():
        if mdcm.IsCustom(flid):
            yield(flid, mdcm.GetOwnClsId(flid), mdcm.GetOwnClsName(flid), mdcm.GetFieldType(flid),
            mdcm.GetFieldName(flid), mdcm.GetFieldWs(flid), mdcm.GetFieldListRoot(flid))
    

customFields = GetAllCustomFields()
for field in customFields :
    print(field[0],field[1],field[2],field[3],field[4],field[5],field[6], sep=", ")

5002500, 5002, LexEntry, 13, SemanticDomainsCheck, -1, 00000000-0000-0000-0000-000000000000
5002501, 5002, LexEntry, 16, SummaryPOS, -3, 00000000-0000-0000-0000-000000000000
5016500, 5016, LexSense, 13, EngReversalCheck, -1, 00000000-0000-0000-0000-000000000000
5016501, 5016, LexSense, 16, PublishedPOS, -3, 00000000-0000-0000-0000-000000000000
5016502, 5016, LexSense, 13, ScratchEng, -1, 00000000-0000-0000-0000-000000000000
5035500, 5035, MoForm, 23, Note, -1, 00000000-0000-0000-0000-000000000000


In [47]:
text_repo = sl.GetService(ITextRepository)
notebook_repo = sl.GetService(IRnGenericRecRepository)

count = 0

# Get all texts using the repository
count = 0
text_para_dict = {}
for text in text_repo.AllInstances():
    count += 1
    if count > 5:
        break
        
    comment = text.Description.get_String(wsa)
    if comment is not None and hasattr(comment, "Text") and comment.Text is not None:
        if comment.Text.lower() == 'y':
            paragraphs = []

            title = text.Name.BestVernacularAnalysisAlternative
            if title != None:
                # Safe to access .Text property
                title_text = title.Text
                if title_text:  # Check if the text content is not empty
                    print(title_text)
            # Access the structured content
            sttext = text.ContentsOA
            tags = sttext.TagsOC
            texts = []

            if sttext is not None and hasattr(sttext, "ParagraphsOS"):
                # print(f"Number of paragraphs: {sttext.ParagraphsOS.Count}")
                # Access baseline text from paragraphs
                for para in sttext.ParagraphsOS:
                    # Cast to IStTxtPara to access Contents
                    if hasattr(para, 'Contents'):
                        # It's already IStTxtPara
                        txtpara = para
                    else:
                        # Cast from IStPara to IStTxtPara
                        txtpara = IStTxtPara(para)
                    paragraphs.append(txtpara)

            text_para_dict[title] = paragraphs
create_docx("testdoc2", text_para_dict)
                
        
print(count)

Expo_Gı`lina umoyzar, q`ı`dimna humoyzar.
Narr_Rooster and the Fox
Narr_Cəfarna k`onayvalla. 
Narr_Zakir's Bear Story
Expo_Gı`lina umoyzar, q`ı`dimna humoyzar. + 2
75382
17694
51606
70407
2266
88685
78675
49497
85921
24288
82513
Narr_Rooster and the Fox + 31
91133
14165
56438
86978
14097
75523
85124
38888
21041
Error with WfiAnalysis: 'ICmObject' object has no attribute 'Form'
9954
67011
80593
47484
96473
83089
75969
88238
72170
87889
69852
26375
55755
35229
71027
16561
69651
63769
85664
84135
85676
80513
94286
2062
Narr_Cəfarna k`onayvalla.  + 11
62459
75141
32670
11570
1991
7666
3946
36749
50069
60843
26732
97943
16075
98214
91463
57752
78301
20442
38907
45641
22753
41206
21262
Narr_Zakir's Bear Story + 35
36110
70234
45685
60300
45582
14716
64855
15770
89535
27853
35281
78314
99135
20399
89647
45252
28394
67312
62246
94582
9121
97110
32249
30125
41490
28120
90230
116
13433
87325
64909
74935
6


In [34]:
def create_docx(doc_name: str, title_para_dict: dict):
    doc = docx.Document()
    doc.sections[0].left_margin = Inches(0.5)
    doc.sections[0].right_margin = Inches(0.5)

    for title, paragraphs in title_para_dict.items():
        # doc.add_heading(title)
        print(title, "+", len(paragraphs))
        for p in paragraphs:
            create_table(p, doc)
            doc.add_page_break()
    
    doc.save(doc_name + ".docx")
    

In [45]:
# with help from Claude
def create_table(txtpara, doc: docx.Document):
    char_width = 0.11  # average width per character in inches
    min_width = 0.4    # minimum cell width in inches
    max_table_width = 7.2  # maximum table width in inches
    
    # Define the row types you want
    row_types = [
        {"label": "Baseline", "merge": False},
        {"label": "Word for Word", "merge": False}, 
        {"label": "Free", "merge": True},
    ]
    
    # if (txtpara.Label is not None):
    #     print(txtpara.Label)

    seg_count = 0
    
    for segment in txtpara.SegmentsOS:
        seg_count += 1
        
        print(segment.Hvo)

        paragraph = doc.add_paragraph()
        run = paragraph.add_run(str(segment.Hvo))
        run.font.size = Pt(0.1)
        
        # for n in segment.NotesOS:
        #     print(n.Content.Text)
        
        tup_forms = []
        for analysis in segment.AnalysesRS:
            tup = get_form(analysis, wsv)
            if tup is not None:
                tup_forms.append(tup)
        
        # Combine forms with punctuation
        forms = []
        for i, (form, punctuation) in enumerate(tup_forms):
            if punctuation:
                if i != 0:
                    forms[-1] = forms[-1] + form
            else:
                forms.append(form)
        
        if not forms:
            continue
            
        # Split forms into chunks that fit within max_table_width
        form_chunks = split_forms_into_chunks(forms, char_width, min_width, max_table_width)
        
        # Create a table for each chunk
        for chunk_idx, chunk_forms in enumerate(form_chunks):
            # Create table with row types
            table = doc.add_table(rows=len(row_types), cols=(len(chunk_forms) + 1))
            table.autofit = False  # We'll control widths manually
            
            # Set first column width for labels
            for row in table.rows:
                row.cells[0].width = Inches(1)
            
            # Set widths for word columns and add baseline text
            for col_idx, form in enumerate(chunk_forms, start=1):
                if form is not None:
                    width = max(len(form) * char_width, min_width)
                    
                    # Set width for all rows in this column
                    for row in table.rows:
                        row.cells[col_idx].width = Inches(width)
                    
                    # Add form text to baseline row (first row)
                    table.rows[0].cells[col_idx].text = form
            
            # Set row labels and handle merging
            for row_idx, row_type in enumerate(row_types):
                # Add label
                table.rows[row_idx].cells[0].text = row_type["label"]
                
                # Merge cells for free translation rows if specified
                if row_type["merge"] and len(chunk_forms) > 0:
                    base_cell = table.rows[row_idx].cells[1]
                    for merge_idx in range(2, len(chunk_forms) + 1):
                        base_cell = base_cell.merge(table.rows[row_idx].cells[merge_idx])
            
            # Add spacing between chunk tables (optional)
            # if chunk_idx < len(form_chunks) - 1:
            #     doc.add_paragraph("")  # Small spacing between continuation tables
        
        doc.add_page_break()

def split_forms_into_chunks(forms, char_width, min_width, max_table_width):
    """
    Split forms into chunks where each chunk fits within max_table_width.
    This uses a column-aware algorithm that maintains alignment.
    """
    if not forms:
        return []
    
    chunks = []
    current_chunk = []
    current_width = min_width  # Start with label column width
    
    for form in forms:
        # Calculate width needed for this form
        form_width = max(len(form) * char_width, min_width) if form else min_width
        
        # Check if adding this form would exceed max width
        if current_chunk and (current_width + form_width > max_table_width):
            # Save current chunk and start new one
            chunks.append(current_chunk)
            current_chunk = [form]
            current_width = min_width + form_width  # Reset with label column + new form
        else:
            # Add to current chunk
            current_chunk.append(form)
            current_width += form_width
    
    # Add the last chunk if not empty
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks

In [ ]:
project.CloseProject();